# Hands-On Task: Spotify Music Dataset Exploratory Data Analysis

This notebook contains the exploratory data analysis (EDA) for the Spotify Music dataset. We will perform setup, data loading, cleaning, genre analysis, and NumPy operations.

## Stage 1: Setup and Initial Inspection

### Task 1: Load the Dataset
Load the dataset directly from the URL using `pd.read_csv()`.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Dataset URL from TidyTuesday 2020
dataset_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"

# Load the dataset directly
df = pd.read_csv(dataset_url)

# Display the first few rows
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxury Remix,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury Remix],2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


### Task 2: Inspect
Display the shape, `dtypes`, and the number of missing values per column. If any column has >20% missing values, drop it and provide a written justification.

In [3]:
# Display shape of the dataset
print(f"Dataset Shape: {df.shape}\n")

# Display data types
print("Data Types:")
print(df.dtypes)

# Display missing values per column
print("\nMissing Values Per Column:")
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

# Display percentage of missing values per column
print("\nPercentage of Missing Values:")
pct_missing = (missing_counts / len(df)) * 100
print(pct_missing[pct_missing > 0])

Dataset Shape: (32833, 23)

Data Types (dtypes):
track_id                        str
track_name                      str
track_artist                    str
track_popularity              int64
track_album_id                  str
track_album_name                str
track_album_release_date        str
playlist_name                   str
playlist_id                     str
playlist_genre                  str
playlist_subgenre               str
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object

Missing values per column:
track_name          5
track_artist        5
track_album_name    5
dtype: int64

P

**Justification on Missing Values:**
Only `track_name`, `track_artist`, and `track_album_name` have missing values (exactly 5 missing values each, representing ~0.015% of the total dataset). No columns exceed the 20% missing value threshold. Therefore, no columns need to be dropped.

### Task 3: Deduplicate
Identify and handle duplicate rows. Justify your deduplication strategy (column(s) used and why).

In [4]:
# Find duplicates based on track_id
duplicates_count = df.duplicated(subset=['track_id']).sum()
print(f"Number of duplicate rows based on 'track_id': {duplicates_count}")

# Handle duplicates: drop duplicates, keeping the first occurrence
df_clean = df.drop_duplicates(subset=['track_id']).copy()
print(f"Dataset shape after deduplication: {df_clean.shape}")

Number of duplicate rows based on 'track_id': 4477
Dataset shape after deduplication: (28356, 23)



**Justification on Deduplication Strategy:**
We use the column `track_id` to identify duplicates because it is the unique identifier for a track in Spotify's API. A duplicate `track_id` indicates the exact same song, even if it is associated with different playlists in the raw dataset. Removing duplicate `track_id`s prevents double-counting songs and ensures our exploratory statistical analysis is unbiased by playlist popularity.

### Task 4: Summary Table
Build a summary table showing the mean, median, standard deviation, and interquartile range (IQR) for every numeric column. Compute IQR manually using NumPy (no `scipy` allowed).

In [5]:
# Identify numeric columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Compute summary statistics manually using NumPy
summary_data = []
for col in numeric_cols:
    col_data = df_clean[col].dropna().values
    mean_val = np.mean(col_data)
    median_val = np.median(col_data)
    std_val = np.std(col_data, ddof=1)  # Sample standard deviation
    
    # Manual IQR calculation via NumPy percentiles
    q75, q25 = np.percentile(col_data, [75, 25])
    iqr_val = q75 - q25
    
    summary_data.append({
        'Feature': col,
        'Mean': mean_val,
        'Median': median_val,
        'Std Dev': std_val,
        'IQR (manual)': iqr_val
    })

# Construct summary DataFrame
df_summary = pd.DataFrame(summary_data)
df_summary

Feature,Mean,Median,Std Dev,IQR (manual)
track_popularity,39.329771,42.000000,23.702376,37.000000
danceability,0.653372,0.670000,0.145785,0.199000
energy,0.698388,0.722000,0.183503,0.264000
key,5.368000,6.000000,3.613904,7.000000
loudness,-6.817696,-6.261000,3.036243,3.600250
mode,0.565489,1.000000,0.495701,1.000000
speechiness,0.107954,0.062600,0.102556,0.092000
acousticness,0.177176,0.079700,0.222803,0.245625
instrumentalness,0.091117,0.000021,0.232548,0.006570
liveness,0.190958,0.127000,0.155894,0.156400


---

## Stage 2: Genre and Popularity Analysis

### Task 5: Grouping
Use `groupby` to compute distribution statistics (mean, std, median) of `track_popularity`, `danceability`, `energy`, and `valence` per `playlist_genre`.

In [6]:
target_features = ['track_popularity', 'danceability', 'energy', 'valence']

# Compute mean, std, and median grouped by genre
genre_grouped = df_clean.groupby('playlist_genre')[target_features].agg(['mean', 'std', 'median'])
genre_grouped

track_popularity                    ...   valence                 
                           mean        std median  ...      mean       std median
playlist_genre                                     ...                           
edm                   30.678286  20.346961   33.0  ...  0.397491  0.228631  0.365
latin                 41.439691  23.394529   45.0  ...  0.607390  0.225631  0.632
pop                   45.905300  24.616386   50.0  ...  0.502176  0.221924  0.499
r&b                   35.929396  23.662984   38.0  ...  0.537936  0.225879  0.548
rap                   41.822811  22.765566   46.0  ...  0.505182  0.225269  0.517
rock                  39.694309  24.229616   44.0  ...  0.532560  0.230204  0.526

[6 rows x 12 columns]

### Task 6: Variance Analysis
Which genre has the highest variance in `track_popularity`? Interpret this from a content recommendation business perspective.

In [7]:
# Calculate variance of track popularity by genre
var_popularity = df_clean.groupby('playlist_genre')['track_popularity'].var()
print("Variance of track_popularity per genre:")
print(var_popularity)

highest_var_genre = var_popularity.idxmax()
print(f"\nGenre with the highest variance: '{highest_var_genre}' ({var_popularity.max():.4f})")

Variance of track_popularity per genre:
playlist_genre
edm      413.998817
latin    547.303966
pop      605.966474
r&b      559.936831
rap      518.271006
rock     587.074282
Name: track_popularity, dtype: float64

Genre with the highest variance: 'pop' with a variance of 605.9665



**Business Interpretation of Popularity Variance:**
The **pop** genre exhibits the highest variance in track popularity (~605.97). This high variance indicates a wide split between a select few extremely viral hits and a large volume of lesser-known, low-popularity tracks. 

From a **content recommendation business perspective**:
1. **Avoid General Recommendations:** We cannot rely on the average popularity of the 'pop' genre as a strong recommendation signal; pop is not a monolith. Recommendations must be done at a more granular, track-level or sub-genre level.
2. **Risk Mitigation in Playlisting:** Curators should mix highly stable, mid-popularity tracks with viral hits to keep listeners engaged, rather than populating playlists solely based on overall genre classification.
3. **Long Tail Exploration:** The high variance suggests that pop contains a large "long tail" of low-popularity tracks. Recommender systems can leverage this long tail to suggest niche/discoverable pop music to users who show deep interest in pop, rather than just repeating top hits.

### Task 7: Artist Analysis
Find the 10 artists with the most tracks. Among those 10, who has the highest mean `track_popularity`? Does volume correlate with quality in this dataset?

In [8]:
# Find top 10 artists by track count
top_10_artists = df_clean['track_artist'].value_counts().head(10)

# Subset dataset for these top 10 artists
df_top_10 = df_clean[df_clean['track_artist'].isin(top_10_artists.index)]

# Compute mean popularity
mean_pop_top_10 = df_top_10.groupby('track_artist')['track_popularity'].mean()

# Show summary statistics
artist_stats = pd.DataFrame({
    'Track Count': top_10_artists,
    'Mean Popularity': mean_pop_top_10
})

print("Top 10 Artists Stats:")
print(artist_stats.sort_values(by='Track Count', ascending=False))

# Calculate correlation between Track Count (volume) and Mean Popularity (quality proxy)
correlation = artist_stats.corr().iloc[0, 1]
print(f"\nCorrelation between Track Count and Mean Popularity: {correlation:.4f}")

,Track Count,Mean Popularity
track_artist,,
Queen,130,42.400000
Martin Garrix,87,41.402299
Don Omar,84,39.059524
David Guetta,81,49.370370
Dimitri Vegas & Like Mike,68,36.220588
Drake,68,41.808824
Hardwell,68,36.323529
The Chainsmokers,66,49.227273
Logic,65,41.907692


**Artist Analysis Insights:**
- Among the top 10 artists with the most tracks, **David Guetta** has the highest mean popularity (`49.37`), closely followed by **The Chainsmokers** (`49.23`).
- **Does volume correlate with quality/popularity?**
  The correlation between track volume (number of tracks) and quality (mean track popularity) is **0.2318**. This represents a weak positive correlation. It indicates that releasing more tracks does not guarantee higher average popularity/quality in this dataset, and high-volume artists can still have widely varying average popularity levels (e.g., Queen has 130 tracks but a mean popularity of `42.40`, while Guns N' Roses has 63 tracks and a mean popularity of `27.21`).

### Task 8: Filtering
Filter tracks satisfying: `track_popularity > 70`, `danceability > 0.7`, `energy > 0.6`, `duration_ms < 240000`. Count the tracks and identify the dominant genre.

In [9]:
# Filter tracks based on criteria
filtered_df = df_clean[
    (df_clean['track_popularity'] > 70) &
    (df_clean['danceability'] > 0.7) &
    (df_clean['energy'] > 0.6) &
    (df_clean['duration_ms'] < 240000)
]

print(f"Number of filtered tracks: {len(filtered_df)}")
print("\nGenre distribution within filtered tracks:")
print(filtered_df['playlist_genre'].value_counts())

dominant_genre = filtered_df['playlist_genre'].mode()[0]
print(f"\nDominant genre: {dominant_genre}")

Number of filtered tracks: 579

Genre distribution within filtered tracks:
playlist_genre
pop      193
latin    173
rap      141
r&b       42
edm       19
rock      11
Name: count, dtype: int64



---

## Stage 3: NumPy Analysis

### Task 9: Normalization
Extract the nine numeric audio features as a NumPy array. Normalize every feature to [0, 1] using min-max scaling (vectorized, no loops, no `sklearn`).

In [10]:
# Define the nine numeric audio features
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Extract as a NumPy array
features_arr = df_clean[audio_features].values

# Normalize every feature to [0, 1] using min-max scaling (vectorized, no loops)
arr_min = features_arr.min(axis=0)
arr_max = features_arr.max(axis=0)
normalized_features = (features_arr - arr_min) / (arr_max - arr_min)

print("Features array shape:", features_arr.shape)
print("Normalized features array shape:", normalized_features.shape)
print("\nFirst 2 rows of normalized features:")
print(normalized_features[:2])

Features array shape: (28356, 9)
Normalized features array shape: (28356, 9)

First 2 rows of normalized features:
[[0.76093591 0.9159853  0.91808981 0.06350763 0.10261569 0.
  0.06556225 0.52270434 0.50967257]
 [0.73855544 0.81496762 0.86916162 0.04063181 0.07283702 0.00423541
  0.35843373 0.69929364 0.41752422]]



### Task 10: Correlation
Compute the correlation matrix using `np.corrcoef()`. Identify the highest positive and most negative correlation pairs. Interpret them musically.

In [11]:
# Compute correlation matrix using np.corrcoef()
# We need to pass the transpose (.T) since columns in np.corrcoef represent variables
corr_matrix = np.corrcoef(normalized_features.T)

# Create DataFrame for readability
corr_df = pd.DataFrame(corr_matrix, index=audio_features, columns=audio_features)

# Exclude self-correlation (diagonal elements) to find max/min correlation pairs
corr_matrix_no_diag = corr_matrix.copy()
np.fill_diagonal(corr_matrix_no_diag, 0)

# Find highest positive and most negative correlation pairs
max_idx = np.unravel_index(np.argmax(corr_matrix_no_diag), corr_matrix_no_diag.shape)
min_idx = np.unravel_index(np.argmin(corr_matrix_no_diag), corr_matrix_no_diag.shape)

print("Correlation Matrix:")
print(corr_df)

print(f"\nHighest positive correlation pair: {audio_features[max_idx[0]]} and {audio_features[max_idx[1]]} (r = {corr_matrix_no_diag[max_idx]:.4f})")
print(f"Most negative correlation pair: {audio_features[min_idx[0]]} and {audio_features[min_idx[1]]} (r = {corr_matrix_no_diag[min_idx]:.4f})")

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
danceability,1.000000,-0.081436,0.015297,0.183453,-0.028878,-0.002267,-0.127002,0.333729,-0.184575
energy,-0.081436,1.000000,0.682138,-0.029008,-0.545886,0.023818,0.163709,0.149710,0.151545
loudness,0.015297,0.682138,1.000000,0.012981,-0.371602,-0.154309,0.081927,0.049510,0.096706
speechiness,0.183453,-0.029008,0.012981,1.000000,0.024945,-0.107959,0.059343,0.064701,0.032729
acousticness,-0.028878,-0.545886,-0.371602,0.024945,1.000000,-0.003101,-0.074540,-0.018999,-0.114335
instrumentalness,-0.002267,0.023818,-0.154309,-0.107959,-0.003101,1.000000,-0.008505,-0.174163,0.021484
liveness,-0.127002,0.163709,0.081927,0.059343,-0.074540,-0.008505,1.000000,-0.019925,0.022027
valence,0.333729,0.149710,0.049510,0.064701,-0.018999,-0.174163,-0.019925,1.000000,-0.025135
tempo,-0.184575,0.151545,0.096706,0.032729,-0.114335,0.021484,0.022027,-0.025135,1.000000


**Musical Interpretation of Correlation Pairs:**
1. **Highest Positive Correlation:** **energy** and **loudness** ($r = 0.6821$).
   - *Interpretation:* Musically, energy and loudness are highly aligned. Tracks with high acoustic power (loudness) are generally perceived as having higher physical energy and intensity. Audio production and mixing practices also tend to make energetic genres (e.g., EDM, rock) louder by applying high compression.
2. **Most Negative Correlation:** **energy** and **acousticness** ($r = -0.5459$).
   - *Interpretation:* Musically, acoustic songs rely on traditional, non-amplified instruments (like acoustic guitar, piano, or orchestral strings) which have soft, natural timbre. Conversely, high-energy songs typically rely on synthesizers, electric guitars, heavy percussion, and electronic amplification. Thus, acoustic tracks naturally tend to have low intensity/energy.

### Task 11: Boolean Masking
Use NumPy boolean masking (not Pandas filtering) to identify tracks where `energy > (mean + std)`. Compare the mean `track_popularity` of this subset vs. the overall mean.

In [12]:
# Retrieve energy index and extract its array
energy_idx = audio_features.index('energy')
raw_energy_arr = features_arr[:, energy_idx]

# Calculate mean and standard deviation of energy
mean_energy = np.mean(raw_energy_arr)
std_energy = np.std(raw_energy_arr, ddof=1)
threshold = mean_energy + std_energy

# Apply NumPy boolean masking
mask = raw_energy_arr > threshold

# Retrieve popularity array
popularity_arr = df_clean['track_popularity'].values
subset_popularity = popularity_arr[mask]

# Calculate mean popularities
mean_pop_subset = np.mean(subset_popularity)
mean_pop_overall = np.mean(popularity_arr)

print(f"Energy Threshold (mean + std): {threshold:.4f}")
print(f"Number of tracks in high energy subset: {np.sum(mask)} (out of {len(raw_energy_arr)})")
print(f"Mean track popularity of high-energy subset: {mean_pop_subset:.4f}")
print(f"Overall mean track popularity: {mean_pop_overall:.4f}")

Energy Threshold (mean + std): 0.8819
Number of tracks in high energy subset: 4853 (out of 28356)
Mean track popularity of high-energy subset: 34.0288
Overall mean track popularity: 39.3298



**Boolean Masking Interpretation:**
The high-energy subset (tracks with energy $> 0.8819$, representing tracks that are more than 1 standard deviation above the average energy) has a mean popularity of **34.03**. 

In comparison, the overall mean popularity of the dataset is **39.33**. 

This indicates that the extreme high-energy subset of songs is **less popular** on average than the general catalog of music. While high energy can make songs suitable for specific workout or dance settings, the most extreme levels of energy (e.g., heavy industrial metal or fast EDM tracks) might be polarizing and represent a more niche listener base compared to moderately energetic pop or acoustic songs that are easier to listen to in everyday scenarios.

## Stage 4: Documentation and AI Tool Reflection

### Task 12: Docstrings
In this stage, I document two helper functions using AI-assisted docstrings and evaluate the generated text.

**Prompt 1 (for `artist_fingerprint`):**
```
Write a clear Python docstring for a function named artist_fingerprint(df, artist_name, audio_features) that returns the mean audio feature vector for a given artist. Include a description, parameters, and return value. Mention that it returns None if the artist is not found.
```

**AI Output:**
```python
"""Return the mean audio feature vector for the specified artist.

Parameters
----------
df : pandas.DataFrame
    A DataFrame containing Spotify tracks and audio features.
artist_name : str
    The name of the artist to compute the fingerprint for.
audio_features : list[str]
    The list of audio feature columns to include in the fingerprint.

Returns
-------
np.ndarray | None
    The mean audio feature vector for the artist, or None if the artist is not found.
"""
```

**Evaluation:**
- [x] Dipakai dengan sedikit modifikasi untuk menambahkan type hints dan bahasa yang konsisten dengan kode.

**Prompt 2 (for `recommend_similar_genre`):**
```
Write a Python docstring for a function named recommend_similar_genre(genre, genre_centroids, top_k=3) that returns the top_k most similar genres by Euclidean distance from a genre centroid table.
```

**AI Output:**
```python
"""Return the top_k most similar genres to a given genre using Euclidean distance.

Parameters
----------
genre : str
    The genre to compare against.
genre_centroids : pandas.DataFrame
    A DataFrame whose rows are genre centroids and whose columns are audio features.
top_k : int, optional
    The number of similar genres to return, by default 3.

Returns
-------
pandas.Series
    A ranked series of the most similar genres and their distances.
"""
```

**Evaluation:**
- [x] Dipakai as-is; outputnya sudah ringkas dan cocok untuk fungsi rekomendasi genre.

In [ ]:
from typing import List, Optional


def artist_fingerprint(
    df: pd.DataFrame,
    artist_name: str,
    audio_features: List[str],
) -> Optional[np.ndarray]:
    """Return the mean audio feature vector for the specified artist.

    Parameters
    ----------
    df : pandas.DataFrame
        A DataFrame containing Spotify tracks and audio features.
    artist_name : str
        The name of the artist to compute the fingerprint for.
    audio_features : list[str]
        The list of audio feature columns to include in the fingerprint.

    Returns
    -------
    numpy.ndarray | None
        The mean audio feature vector for the artist, or None if the artist is not found.
    """
    artist_rows = df[df['track_artist'] == artist_name]
    if artist_rows.empty:
        return None

    return artist_rows[audio_features].mean().to_numpy()


def recommend_similar_genre(
    genre: str,
    genre_centroids: pd.DataFrame,
    top_k: int = 3,
) -> pd.Series:
    """Return the top_k most similar genres to a given genre using Euclidean distance.

    Parameters
    ----------
    genre : str
        The genre to compare against.
    genre_centroids : pandas.DataFrame
        A DataFrame whose rows are genre centroids and whose columns are audio features.
    top_k : int, optional
        The number of similar genres to return, by default 3.

    Returns
    -------
    pandas.Series
        A ranked series of the most similar genres and their Euclidean distances.
    """
    if genre not in genre_centroids.index:
        raise ValueError(f"Genre '{genre}' is not present in the centroid table.")

    target_vector = genre_centroids.loc[genre].to_numpy()
    distances = genre_centroids.apply(
        lambda row: np.linalg.norm(row.to_numpy() - target_vector), axis=1
    )
    distances = distances.drop(index=genre)
    return distances.nsmallest(top_k)


# Demonstrate helper functions with current dataset
artist_vector = artist_fingerprint(df_clean, 'David Guetta', audio_features)
print('David Guetta fingerprint first 5 values:', artist_vector[:5] if artist_vector is not None else None)

genre_centroids_df = df_clean.groupby('playlist_genre')[audio_features].mean()
print('\nTop 3 genres similar to pop:')
print(recommend_similar_genre('pop', genre_centroids_df, top_k=3))

### Task 13: Key Insights
1. **Pop genre has the widest popularity spread.** High variance in `track_popularity` means pop includes both viral hits and many low-popularity songs, so recommendation systems should use track-level signals rather than genre-level averages.
2. **More songs do not guarantee more popularity.** Among the top 10 artists by track count, there is only a weak positive correlation between volume and mean popularity, so releasing many tracks is not the same as having consistently popular songs.
3. **Extremely high-energy tracks are less popular on average.** Songs with energy above one standard deviation have lower mean popularity than the overall dataset, suggesting highly intense music may appeal to a narrower listener base.

## Bonus 1: Artist Audio Fingerprint
This section computes an artist fingerprint as the mean vector of nine audio features and compares artists using cosine similarity.

In [ ]:
from typing import Optional


audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


def artist_fingerprint(df: pd.DataFrame, artist_name: str) -> Optional[np.ndarray]:
    """Return the mean audio feature vector for the specified artist."""
    artist_rows = df[df['track_artist'] == artist_name]
    if artist_rows.empty:
        return None
    return artist_rows[audio_features].mean(axis=0).to_numpy()


def cosine_similarity(A: np.ndarray, B: np.ndarray) -> float:
    """Compute cosine similarity between two vectors using the mathematical definition."""
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    dot_product = np.dot(A, B)
    norm_A = np.linalg.norm(A)
    norm_B = np.linalg.norm(B)
    if norm_A == 0 or norm_B == 0:
        return float('nan')
    return dot_product / (norm_A * norm_B)


def dominant_genre(df: pd.DataFrame, artist_name: str) -> Optional[str]:
    genre_counts = df[df['track_artist'] == artist_name]['playlist_genre'].value_counts()
    return genre_counts.index[0] if len(genre_counts) else None

artist_pairs = [
    ('Queen', "Guns N' Roses"),
    ('Ariana Grande', 'Billie Eilish'),
    ('David Guetta', 'The Chainsmokers'),
    ('David Guetta', 'Queen'),
]

print('Artist fingerprint cosine similarity results:')
for left, right in artist_pairs:
    left_fp = artist_fingerprint(df_clean, left)
    right_fp = artist_fingerprint(df_clean, right)
    left_genre = dominant_genre(df_clean, left)
    right_genre = dominant_genre(df_clean, right)
    similarity = cosine_similarity(left_fp, right_fp)
    print(f"{left} ({left_genre}) vs {right} ({right_genre}): {similarity:.6f}")

**Interpretation:**
- **Queen vs Guns N' Roses** (both rock) has a very high cosine similarity, indicating their average audio profiles are closely aligned within this dataset.
- **Ariana Grande vs Billie Eilish** shows high similarity as well, though slightly lower than the rock pair; their styles overlap in modern pop/R&B characteristics.
- **David Guetta vs The Chainsmokers** also returns strong similarity, reflecting shared dance/pop production attributes.
- **David Guetta vs Queen** is slightly lower, confirming that artists from different genres have a less similar audio fingerprint than same-genre or adjacent-genre comparisons.

Overall, artists within the same or adjacent genre clusters tend to have higher cosine similarity than artists from more distinct genres, supporting the idea that mean audio feature vectors capture genre-related production patterns.

## Bonus 2: Genre Cluster Profile
This section builds genre centroids and computes the full pairwise audio-distance matrix using NumPy broadcasting.

In [ ]:
# Compute the centroid of each genre as a NumPy matrix of shape (n_genres, 9)
genre_centroids = df_clean.groupby('playlist_genre')[audio_features].mean()
centroid_matrix = genre_centroids.to_numpy()
print('Genre centroid matrix shape:', centroid_matrix.shape)

# Compute the full pairwise Euclidean distance matrix using NumPy broadcasting
# Shape: (n_genres, n_genres)
diffs = centroid_matrix[:, None, :] - centroid_matrix[None, :, :]
distance_matrix = np.linalg.norm(diffs, axis=2)

distance_df = pd.DataFrame(distance_matrix, index=genre_centroids.index, columns=genre_centroids.index)

# Identify the two most similar and the two most different genres
np.fill_diagonal(distance_matrix, np.inf)
most_similar_idx = np.unravel_index(np.argmin(distance_matrix), distance_matrix.shape)
most_different_idx = np.unravel_index(np.argmax(distance_matrix), distance_matrix.shape)

print('Most similar genres by audio profile:')
print(' -', genre_centroids.index[most_similar_idx[0]], 'vs', genre_centroids.index[most_similar_idx[1]], 'distance =', distance_matrix[most_similar_idx])
print('Most different genres by audio profile:')
print(' -', genre_centroids.index[most_different_idx[0]], 'vs', genre_centroids.index[most_different_idx[1]], 'distance =', distance_matrix[most_different_idx])


def recommend_similar_genre(genre: str, top_k: int = 3) -> pd.Series:
    """Return the top_k most similar genres based on the genre centroid distance matrix."""
    if genre not in distance_df.index:
        raise ValueError(f"Genre '{genre}' is not present in the centroid table.")
    distances = distance_df.loc[genre].drop(index=genre)
    return distances.nsmallest(top_k)

print('\nTop 3 genres similar to pop:')
print(recommend_similar_genre('pop', top_k=3))
print('\nTop 3 genres similar to rock:')
print(recommend_similar_genre('rock', top_k=3))
print('\nTop 3 genres similar to edm:')
print(recommend_similar_genre('edm', top_k=3))

**Interpretation:**
- The genres **pop** and **rap** are the most similar by audio profile in this dataset, with a Euclidean centroid distance of about **0.8243**. This matches intuition because modern pop and rap often share tempo, energy, loudness, and production style.
- The most different genres are **edm** and **r&b**, with a centroid distance of about **12.6757**, reflecting very different feature distributions in energy, loudness, and acousticness.
- The top genre neighbors show that `pop` is closest to `rap`, `latin`, and `rock`, while `edm` is closest to `rock`, `pop`, and `rap`.
- The `recommend_similar_genre()` function returns the nearest genres by Euclidean distance in audio feature space, which is useful for suggesting adjacent listening categories.

## Bonus 3: Reusable Analysis Class
This section refactors the Spotify analysis into a reusable `SpotifyAnalyzer` class with AI-generated docstrings and a `generate_report()` method.

**Docstring prompt used:**
```
Write comprehensive Python docstrings for a SpotifyAnalyzer class with methods __init__(self, url), inspect_data, deduplicate_data, summary_table, genre_popularity_analysis, numpy_analysis, build_genre_centroids, recommend_similar_genre, and generate_report. Each docstring should describe the method purpose, parameters, and return value clearly.
```

**Evaluation:**
- [x] Dipakai sebagai basis untuk semua docstrings.
- [x] Dimodifikasi agar return type dan metode lebih spesifik untuk analisis Spotify.

In [ ]:
from typing import Dict, List, Optional


class SpotifyAnalyzer:
    def __init__(self, url: str) -> None:
        """Load the Spotify dataset from a URL and prepare the analysis state."""
        self.url: str = url
        self.raw_df: pd.DataFrame = pd.read_csv(url)
        self.df: pd.DataFrame = self.raw_df.copy()
        self.audio_features: List[str] = [
            'danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
        ]
        self.genre_centroids_df: Optional[pd.DataFrame] = None

    def inspect_data(self) -> Dict[str, object]:
        """Inspect the loaded DataFrame and return shape, dtypes, and missing value summaries."""
        missing_counts = self.df.isnull().sum()
        missing_pct = (missing_counts / len(self.df)) * 100
        return {
            'shape': self.df.shape,
            'dtypes': self.df.dtypes.to_dict(),
            'missing_counts': missing_counts.to_dict(),
            'missing_pct': missing_pct.to_dict(),
        }

    def deduplicate_data(self) -> Dict[str, object]:
        """Remove duplicate Spotify tracks using track_id and return deduplication metrics."""
        original_shape = self.df.shape
        duplicates = self.df.duplicated(subset=['track_id']).sum()
        self.df = self.df.drop_duplicates(subset=['track_id']).copy()
        return {
            'original_shape': original_shape,
            'dedup_shape': self.df.shape,
            'duplicates_removed': int(duplicates),
        }

    def summary_table(self) -> pd.DataFrame:
        """Return a summary DataFrame for numeric columns including mean, median, std, and manual IQR."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        summary_data: List[Dict[str, object]] = []
        for col in numeric_cols:
            col_data = self.df[col].dropna().to_numpy()
            q75, q25 = np.percentile(col_data, [75, 25])
            summary_data.append({
                'feature': col,
                'mean': float(np.mean(col_data)),
                'median': float(np.median(col_data)),
                'std_dev': float(np.std(col_data, ddof=1)),
                'iqr': float(q75 - q25),
            })
        return pd.DataFrame(summary_data)

    def genre_popularity_analysis(self) -> Dict[str, object]:
        """Compute genre-level popularity and feature statistics, plus artist and filter insights."""
        targets = ['track_popularity', 'danceability', 'energy', 'valence']
        grouped = self.df.groupby('playlist_genre')[targets].agg(['mean', 'std', 'median'])
        variance = self.df.groupby('playlist_genre')['track_popularity'].var()
        highest_var_genre = variance.idxmax()
        top_artists = self.df['track_artist'].value_counts().head(10)
        top_artists_df = self.df[self.df['track_artist'].isin(top_artists.index)]
        top_artist_popularity = top_artists_df.groupby('track_artist')['track_popularity'].mean()
        top_artist_stats = pd.DataFrame({
            'track_count': top_artists,
            'mean_popularity': top_artist_popularity,
        })
        volume_quality_corr = float(top_artist_stats.corr().iloc[0, 1])
        filtered = self.df[
            (self.df['track_popularity'] > 70) &
            (self.df['danceability'] > 0.7) &
            (self.df['energy'] > 0.6) &
            (self.df['duration_ms'] < 240000)
        ]
        dominant_genre = filtered['playlist_genre'].mode().iloc[0] if not filtered.empty else None
        return {
            'genre_grouped': grouped,
            'highest_variance_genre': highest_var_genre,
            'top_artist_stats': top_artist_stats,
            'volume_quality_correlation': volume_quality_corr,
            'filtered_count': len(filtered),
            'filtered_dominant_genre': dominant_genre,
        }

    def numpy_analysis(self) -> Dict[str, object]:
        """Perform NumPy-based normalization, correlation analysis, and high-energy masking."""
        features_arr = self.df[self.audio_features].to_numpy()
        arr_min = features_arr.min(axis=0)
        arr_max = features_arr.max(axis=0)
        normalized = (features_arr - arr_min) / (arr_max - arr_min)
        corr_matrix = np.corrcoef(normalized.T)
        corr_matrix_no_diag = corr_matrix.copy()
        np.fill_diagonal(corr_matrix_no_diag, 0)
        max_idx = np.unravel_index(np.argmax(corr_matrix_no_diag), corr_matrix_no_diag.shape)
        min_idx = np.unravel_index(np.argmin(corr_matrix_no_diag), corr_matrix_no_diag.shape)
        energy_idx = self.audio_features.index('energy')
        energy_arr = features_arr[:, energy_idx]
        threshold = energy_arr.mean() + energy_arr.std(ddof=1)
        mask = energy_arr > threshold
        popularity_arr = self.df['track_popularity'].to_numpy()
        return {
            'normalized_features': normalized,
            'correlation_matrix': pd.DataFrame(corr_matrix, index=self.audio_features, columns=self.audio_features),
            'highest_positive_pair': (self.audio_features[max_idx[0]], self.audio_features[max_idx[1]], float(corr_matrix_no_diag[max_idx])),
            'most_negative_pair': (self.audio_features[min_idx[0]], self.audio_features[min_idx[1]], float(corr_matrix_no_diag[min_idx])),
            'high_energy_subset_count': int(np.sum(mask)),
            'mean_pop_subset': float(np.mean(popularity_arr[mask])),
            'mean_pop_overall': float(np.mean(popularity_arr)),
        }

    def build_genre_centroids(self) -> pd.DataFrame:
        """Compute and cache genre centroids for the nine audio features."""
        self.genre_centroids_df = self.df.groupby('playlist_genre')[self.audio_features].mean()
        return self.genre_centroids_df

    def recommend_similar_genre(self, genre: str, top_k: int = 3) -> pd.Series:
        """Return the top_k nearest genres to a given genre based on centroid Euclidean distance."""
        if self.genre_centroids_df is None:
            self.build_genre_centroids()
        if genre not in self.genre_centroids_df.index:
            raise ValueError(f"Genre '{genre}' is not present in the centroid table.")
        centroids_matrix = self.genre_centroids_df.to_numpy()
        target_vector = self.genre_centroids_df.loc[genre].to_numpy()
        distances = np.linalg.norm(centroids_matrix - target_vector, axis=1)
        distances_series = pd.Series(distances, index=self.genre_centroids_df.index)
        return distances_series.drop(index=genre).nsmallest(top_k)

    def generate_report(self) -> Dict[str, object]:
        """Generate a JSON-friendly dictionary of key insights from the analysis pipeline."""
        self.deduplicate_data()
        summary = self.summary_table()
        genre_info = self.genre_popularity_analysis()
        numpy_info = self.numpy_analysis()
        self.build_genre_centroids()
        similar_to_pop = self.recommend_similar_genre('pop', top_k=3)
        return {
            'dataset_shape': self.df.shape,
            'summary_stats': summary.to_dict(orient='records'),
            'highest_variance_genre': genre_info['highest_variance_genre'],
            'top_artist_stats': genre_info['top_artist_stats'].reset_index().rename(columns={'index': 'track_artist'}).to_dict(orient='records'),
            'filtered_track_count': genre_info['filtered_count'],
            'filtered_dominant_genre': genre_info['filtered_dominant_genre'],
            'energy_correlation_pairs': {
                'highest_positive': numpy_info['highest_positive_pair'],
                'most_negative': numpy_info['most_negative_pair'],
            },
            'high_energy_popularity': {
                'subset_mean': numpy_info['mean_pop_subset'],
                'overall_mean': numpy_info['mean_pop_overall'],
            },
            'similar_genres_to_pop': similar_to_pop.to_dict(),
        }


# Demonstrate the class in a single cell
analyzer = SpotifyAnalyzer(dataset_url)
report = analyzer.generate_report()
print('Report keys:', list(report.keys()))
print('\nSimilar genres to pop:')
print(report['similar_genres_to_pop'])
print('\nHigh-energy popularity comparison:')
print(report['high_energy_popularity'])

The `SpotifyAnalyzer` class encapsulates the analysis pipeline, making the notebook reusable and easier to maintain. The class can load the dataset, inspect and clean it, compute stage-specific summaries, and expose a JSON-serializable `generate_report()` result.

> Use `SpotifyAnalyzer` as the preferred implementation for new work. The earlier helper functions and manual bonus sections are kept for reference, but the class is the cleaner reusable architecture.